# Gabarito — Treino Unidade 2

Tente fazer o `treino.ipynb` **antes** de olhar aqui. Use este caderno para conferir.

## Preparação (só rodar)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="colorblind")

# bases das aulas 8 e 9
mtcars = pd.read_csv("../Aula_09/data/mtcars.csv")
penguins = pd.read_csv("../Aula_08/data/penguins.csv")
titanic = pd.read_csv("../Aula_08/data/titanic.csv")

# imóveis da Aula 6 já limpos: colunas renomeadas, sem duplicatas, sem ausentes em hoa/parking_spaces
# e com a coluna property_type ("casa"/"apartamento"). floor só é vazio nas casas (casa não tem andar)
imoveis = pd.read_csv("data/imoveis_limpo.csv")


# resume o resultado de um teste em uma linha legível
def report(name, result, stat="estatística"):
    parts = [f"{stat} = {result.statistic:.3f}"]
    if hasattr(result, "df"):
        parts.append(f"gl = {result.df:.4g}")
    p = result.pvalue
    parts.append("p < 0.001" if p < 0.001 else f"p = {p:.3f}")
    print(f"{name}: " + " · ".join(parts))

## Parte 1 — Inferência: permutação e testes paramétricos

### Exercício 1: teste de permutação (Aula 7)

No `mtcars`, a coluna `vs` indica o formato do motor: `vs = 1` é motor **em linha (reto)** e `vs = 0` é motor **em V**.

**Pergunta:** carros com motor reto e com motor em V têm consumo (`mpg`) médio diferente?

a) Escreva H₀ e H₁ (em uma célula de texto ou como comentário).

b) Calcule a diferença observada: `Δobs = média(mpg | vs=1) − média(mpg | vs=0)`.

c) Faça **10.000 permutações** dos rótulos e calcule o p-valor bilateral.

d) Faça o histograma da distribuição nula, com linhas verticais em `+Δobs` e `−Δobs`.

💡 Dica: copie a função `permutation_null` da Aula 7. Ela precisa dos valores e dos rótulos como arrays do NumPy (`.to_numpy()`), com rótulos True/False. p-valor = `np.mean(np.abs(null) >= abs(obs))`.

In [ ]:
# a) H0: o formato do motor não importa -> médias de mpg iguais (Δ = 0)
#    H1: as médias são diferentes (Δ ≠ 0) -> teste bilateral

In [ ]:
# b) diferença observada
reto = mtcars.loc[mtcars["vs"] == 1, "mpg"]
em_v = mtcars.loc[mtcars["vs"] == 0, "mpg"]

delta_obs = reto.mean() - em_v.mean()
round(delta_obs, 2)

In [ ]:
# c) distribuição nula por permutação (função da Aula 7)
def permutation_null(values, labels, n_perm=10_000, seed=2026):
    rng = np.random.default_rng(seed)
    null = []
    for _ in range(n_perm):
        shuffled = rng.permutation(labels)       # embaralha quem é "reto" e quem é "em V"
        null.append(values[shuffled].mean() - values[~shuffled].mean())
    return np.array(null)


values = mtcars["mpg"].to_numpy()
labels = (mtcars["vs"] == 1).to_numpy()          # True = motor reto

vs_null = permutation_null(values, labels)

# p-valor bilateral: fração das permutações tão ou mais extremas que o observado
vs_p = np.mean(np.abs(vs_null) >= abs(delta_obs))
vs_p

In [ ]:
# d) histograma da distribuição nula com o Δobs marcado dos dois lados
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(vs_null, bins=50, color="#A1C9F4", ax=ax)
for value in [-delta_obs, delta_obs]:
    ax.axvline(value, color="#DD8452", linewidth=2)
ax.set(title="Motor reto × em V: Δobs na distribuição do acaso",
       xlabel="Δ de médias com rótulos embaralhados (mpg)", ylabel="Permutações")
plt.tight_layout()
plt.show()

**Em palavras:** o p-valor deu 0. Isso quer dizer que a probabilidade é exatamente zero? Qual a conclusão?

Não. Quer dizer que **nenhuma** das 10.000 permutações chegou a uma diferença tão grande quanto a observada (≈ 7,9 mpg). O certo é dizer **p < 1/10.000 = 0,0001**: com mais permutações talvez aparecesse alguma. No histograma, as linhas laranja ficam muito fora da "montanha" do acaso. **Rejeitamos H₀**: carros com motor reto fazem, em média, cerca de 7,9 mpg a mais, e essa diferença dificilmente seria só sorte.

⚠️ Isso não prova que o motor *causa* a economia. Motores retos nesta base são, em geral, de carros menores e mais leves.

### Exercício 2: teste t de uma amostra

Um guia de campo diz que pinguins Adelie têm bico (`bill_len`) com **39 mm** em média. Os dados confirmam?

a) Escreva H₀ e H₁.

b) Calcule **à mão** o t: `t = (média − 39) / (s / √n)`, e o p-valor com `2 * stats.t.sf(abs(t), df=n - 1)`.

c) Confira com `stats.ttest_1samp` e mostre o **intervalo de confiança de 95%** (`.confidence_interval()`).

💡 Dica: lembre de remover os ausentes (`.dropna()`) antes de calcular.

In [ ]:
# a) H0: μ = 39 mm (o guia está certo)
#    H1: μ ≠ 39 mm

In [ ]:
# b) passo a passo
bill = penguins.loc[penguins["species"] == "Adelie", "bill_len"].dropna()
n = len(bill)
se = bill.std() / np.sqrt(n)               # erro-padrão da média
t = (bill.mean() - 39) / se                # quantos erros-padrão a média está longe de 39
p = 2 * stats.t.sf(abs(t), df=n - 1)       # área das duas caudas da distribuição t

print(f"n = {n} · média = {bill.mean():.2f} · t = {t:.3f} · p = {p:.3f}")

In [ ]:
# c) com o scipy (deve dar o mesmo t e p)
res = stats.ttest_1samp(bill, popmean=39)
report("t de uma amostra", res, stat="t")

ci = res.confidence_interval()
print(f"IC 95% da média: [{ci.low:.2f} ; {ci.high:.2f}] mm")

**Em palavras:** rejeitamos H₀? Isso prova que a média é 39 mm?

**Não rejeitamos H₀** (p ≈ 0,34). A média amostral foi 38,79 mm, só 0,2 mm abaixo de 39, e essa diferença é compatível com o acaso. O IC 95% (~38,4 a 39,2 mm) **contém o 39**.

Isso **não prova** que μ = 39. Os dados são compatíveis com 39 e também com qualquer outro valor dentro do IC, como 38,5. "Não rejeitar" significa que não houve evidência contra, não que H₀ foi confirmada.

### Exercício 3: dois grupos, checando as premissas antes (Aula 8)

O comprimento da nadadeira (`flipper_len`) difere entre pinguins **Adelie** e **Chinstrap**?

a) Separe os dois grupos (sem ausentes) e mostre `n`, média e desvio-padrão de cada um.

b) **Normalidade:** Q-Q plot de cada grupo (lado a lado) + Shapiro-Wilk de cada grupo.

c) **Variâncias:** teste de Levene.

d) Com base em (b) e (c), escolha o teste (t clássico ou Welch) e aplique.

💡 Dica: `stats.probplot(valores, dist="norm", plot=ax)`, `stats.shapiro(...)`, `stats.levene(a, b)`. O t clássico é `stats.ttest_ind(a, b)` e o Welch é `stats.ttest_ind(a, b, equal_var=False)`.

In [ ]:
# a) separa os grupos e resume
flip = penguins.dropna(subset=["flipper_len"])
adelie = flip.loc[flip["species"] == "Adelie", "flipper_len"]
chinstrap = flip.loc[flip["species"] == "Chinstrap", "flipper_len"]

(
    flip
    .query("species in ['Adelie', 'Chinstrap']")
    .groupby("species")
    .flipper_len
    .agg(n="size", média="mean", dp="std")
    .round(2)
)

In [ ]:
# b) normalidade: Q-Q plot + Shapiro de cada grupo
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, values, title in zip(axes, [adelie, chinstrap], ["Adelie", "Chinstrap"]):
    stats.probplot(values, dist="norm", plot=ax)    # pontos perto da reta = parece normal
    ax.set(title=title, xlabel="Quantil teórico da normal", ylabel="Quantil da amostra")
plt.tight_layout()
plt.show()

report("Shapiro Adelie", stats.shapiro(adelie), stat="W")
report("Shapiro Chinstrap", stats.shapiro(chinstrap), stat="W")

In [ ]:
# c) Levene: H0 = variâncias iguais
report("Levene", stats.levene(adelie, chinstrap))

In [ ]:
# d) normalidade OK e variâncias iguais OK -> t clássico é adequado
#    (o Welch também valeria; com variâncias parecidas os dois dão quase o mesmo)
report("t clássico", stats.ttest_ind(chinstrap, adelie), stat="t")
report("t de Welch", stats.ttest_ind(chinstrap, adelie, equal_var=False), stat="t")
print(f"diferença de médias = {chinstrap.mean() - adelie.mean():.2f} mm")

**Em palavras:** as premissas foram atendidas? Qual a conclusão?

- **Normalidade:** os pontos dos Q-Q plots seguem a reta, e o Shapiro **não rejeita** a normalidade em nenhum grupo (p ≈ 0,72 e 0,81).
- **Variâncias:** o Levene **não rejeita** variâncias iguais (p ≈ 0,43).
- As duas premissas valem, então o **t clássico** é adequado (o Welch dá praticamente o mesmo resultado).
- **Conclusão:** p < 0,001, então **rejeitamos H₀**. Pinguins Chinstrap têm nadadeira em média ~5,9 mm maior que os Adelie.

### Exercício 4: teste t pareado

Uma turma de 12 alunos fez uma prova antes e outra depois de um mês de monitoria. Rode a célula com os dados:

In [ ]:
# notas dos MESMOS 12 alunos, antes e depois da monitoria (aluno i está na posição i nas duas listas)
notas = pd.DataFrame({
    "aluno": range(1, 13),
    "antes":  [5.2, 6.1, 4.8, 7.0, 5.5, 6.3, 4.9, 5.8, 6.7, 5.1, 6.0, 5.4],
    "depois": [5.9, 6.4, 5.6, 7.1, 6.3, 6.2, 5.8, 6.5, 7.4, 5.6, 6.9, 5.8],
})
notas

a) Crie a coluna `D = depois − antes` e calcule a média de `D`.

b) Aplique o **teste t pareado** (`stats.ttest_rel`) e, para comparar, o teste **ignorando o pareamento** (`stats.ttest_ind`).

c) Com só 12 alunos, cheque a normalidade de `D` com Shapiro. Aplique também o **Wilcoxon** (`stats.wilcoxon(D)`), a alternativa não paramétrica do pareado.

💡 Dica: no `ttest_rel`, a ordem importa para o sinal do t: `stats.ttest_rel(depois, antes)`.

In [ ]:
# a) a diferença DENTRO de cada aluno
notas = notas.assign(D=lambda df: df["depois"] - df["antes"])
print(f"média de D = {notas['D'].mean():.3f} pontos")
notas

In [ ]:
# b) pareado (certo) x independente (errado, ignora que são os mesmos alunos)
report("t pareado", stats.ttest_rel(notas["depois"], notas["antes"]), stat="t")
report("t ignorando o pareamento", stats.ttest_ind(notas["depois"], notas["antes"]), stat="t")

In [ ]:
# c) premissa do pareado: as DIFERENÇAS devem ser aproximadamente normais
report("Shapiro de D", stats.shapiro(notas["D"]), stat="W")
report("Wilcoxon", stats.wilcoxon(notas["D"]), stat="T")

**Em palavras:** por que o pareado dá um p-valor tão menor que o teste que ignora o pareamento, se a diferença média é a mesma?

Os alunos são muito diferentes entre si: há quem tire ~5 e quem tire ~7. No teste independente, essa variação **entre alunos** entra no erro-padrão e "esconde" boa parte do ganho, e o p fica no limite (≈ 0,048, por pouco abaixo de 0,05). O pareado olha só a diferença **dentro de cada aluno** (`D`). Assim a variação entre alunos se cancela e sobra só a mudança causada pela monitoria, que é bem consistente: 11 dos 12 alunos melhoraram. Por isso o p do pareado fica muito menor (p < 0,001, com t ≈ 6,0 contra 2,1). O Wilcoxon chega à mesma conclusão, o que reforça o resultado mesmo que a normalidade de `D` fosse duvidosa.

**Regra:** se são os **mesmos indivíduos medidos duas vezes**, use o teste **pareado**.

## Parte 2 — Testes não paramétricos

### Exercício 5: Mann-Whitney com dados muito assimétricos

No `titanic`, quem sobreviveu (`survived = 1`) tinha pago uma tarifa (`fare`) diferente de quem não sobreviveu?

a) Mostre média, **mediana** e desvio-padrão da tarifa de cada grupo.

b) Faça um histograma (ou boxplot) da tarifa por grupo e calcule a **assimetria** de cada grupo. Rode o Shapiro também.

c) Aplique o **Welch** e o **Mann-Whitney** (`stats.mannwhitneyu`).

d) Calcule o **δ de Cliff**: `δ = 2·U / (n₁·n₂) − 1`, onde U é a estatística que o `mannwhitneyu` devolve para o **primeiro** grupo.

💡 Dica: `sns.boxplot(data=titanic, x="survived", y="fare")`. Para o δ, `res.statistic` é o U.

In [ ]:
# a) resumo por grupo
(
    titanic
    .groupby("survived")
    .fare
    .agg(n="size", média="mean", mediana="median", dp="std")
    .round(2)
)

In [ ]:
# b) forma da distribuição
sobrev = titanic.loc[titanic["survived"] == 1, "fare"]
morreu = titanic.loc[titanic["survived"] == 0, "fare"]

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.boxplot(data=titanic, x="survived", y="fare", ax=ax)
ax.set(xticks=[0, 1], xticklabels=["Não sobreviveu", "Sobreviveu"], xlabel="", ylabel="Tarifa")
plt.tight_layout()
plt.show()

for name, values in [("sobreviveu", sobrev), ("não sobreviveu", morreu)]:
    print(f"{name}: assimetria = {stats.skew(values):.2f}")
report("Shapiro sobreviveu", stats.shapiro(sobrev), stat="W")
report("Shapiro não sobreviveu", stats.shapiro(morreu), stat="W")

In [ ]:
# c) os dois testes
report("t de Welch", stats.ttest_ind(sobrev, morreu, equal_var=False), stat="t")
mw = stats.mannwhitneyu(sobrev, morreu)
report("Mann-Whitney", mw, stat="U")

In [ ]:
# d) δ de Cliff: vai de -1 a +1; positivo = o 1º grupo (sobreviventes) tende a pagar mais
n1, n2 = len(sobrev), len(morreu)
delta = 2 * mw.statistic / (n1 * n2) - 1
print(f"δ de Cliff = {delta:.2f}")
# outra leitura: P(um sobrevivente sorteado pagou mais que um não sobrevivente sorteado)
print(f"P(sobrevivente pagou mais) ≈ {(1 + delta) / 2:.2f}")

**Em palavras:** qual teste é o mais adequado aqui e por quê? O que significa o δ encontrado?

- A tarifa é **extremamente assimétrica** (cauda longa à direita, com passagens de mais de 500) e o Shapiro rejeita a normalidade com folga. A média é puxada pelos extremos: compare média e mediana em cada grupo.
- Com n grande (~340 e ~550), o Welch até "funciona" pelo Teorema Central do Limite, mas compara **médias** dominadas por poucas passagens caríssimas. O **Mann-Whitney é o mais adequado**: usa postos, não liga para os extremos e responde "um grupo tende a pagar mais que o outro?".
- Os dois rejeitam H₀ (p < 0,001): sobreviventes pagavam tarifas maiores, ou seja, a 1ª classe teve mais sobreviventes.
- O **δ de Cliff ≈ 0,38** é um efeito **médio** (referência comum: |δ| ≈ 0,15 pequeno, 0,33 médio, 0,47 grande). Sorteando um sobrevivente e um não sobrevivente, o sobrevivente pagou mais em ~69% das vezes. A diferença é real, mas os grupos se sobrepõem bastante.

### Exercício 6: Pearson × Spearman

a) Na base `imoveis`, calcule Pearson e Spearman entre `area` e `rent_amount`.

b) Ainda em `imoveis`, calcule Pearson e Spearman entre `area` e `hoa` (só os imóveis com `hoa` > 0, sem ausentes).

c) Faça os gráficos de dispersão das duas relações lado a lado. Use `alpha=.1` e `s=5`, porque são muitos pontos.

💡 Dica: `stats.pearsonr(x, y)` e `stats.spearmanr(x, y)`. Para filtrar: `imoveis.query("hoa > 0")` (o `query` já descarta os NaN nessa comparação).

In [ ]:
# a) área x aluguel
report("Pearson area × rent", stats.pearsonr(imoveis["area"], imoveis["rent_amount"]), stat="r")
report("Spearman area × rent", stats.spearmanr(imoveis["area"], imoveis["rent_amount"]), stat="ρ")

In [ ]:
# b) área x condomínio (só quem paga condomínio)
com_hoa = imoveis.query("hoa > 0")
report("Pearson area × hoa", stats.pearsonr(com_hoa["area"], com_hoa["hoa"]), stat="r")
report("Spearman area × hoa", stats.spearmanr(com_hoa["area"], com_hoa["hoa"]), stat="ρ")

In [ ]:
# c) dispersões lado a lado
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (data, col, ylabel) in zip(axes, [(imoveis, "rent_amount", "Aluguel (R$)"),
                                          (com_hoa, "hoa", "Condomínio (R$)")]):
    ax.scatter(data["area"], data[col], alpha=.1, s=5)
    ax.set(xlabel="Área (m²)", ylabel=ylabel,
           title=f"Pearson {stats.pearsonr(data['area'], data[col]).statistic:.2f} · "
                 f"Spearman {stats.spearmanr(data['area'], data[col]).statistic:.2f}")
plt.tight_layout()
plt.show()

**Em palavras:** nas duas relações, Pearson e Spearman concordam ou divergem? O que isso diz sobre a forma de cada relação?

- **Área × aluguel:** Pearson = 0,72 e Spearman = 0,73.
- **Área × condomínio:** Pearson = 0,85 e Spearman = 0,84.
- Nas duas relações os coeficientes **praticamente coincidem**. Quando isso acontece, a relação monotônica também é bem descrita por uma **reta**: não há curvatura forte nem outliers dominando o Pearson. É o mesmo caso de peso × consumo no mtcars (Aula 9), e o contrário de potência × consumo, onde o Spearman ficou bem acima.
- As duas relações são positivas e fortes, e o condomínio acompanha a área mais de perto (0,85) do que o aluguel (0,72). Faz sentido: o condomínio costuma ser cobrado proporcionalmente ao tamanho, enquanto o aluguel depende também de bairro, padrão, mobília etc.
- Os gráficos mostram que o espalhamento cresce com a área (imóveis grandes variam mais de preço). Isso não muda os coeficientes, mas é bom notar.

### Exercício 7: qual teste usar? (sem código)

Para cada situação, diga **qual teste** você usaria e **por quê**, em uma linha.

a) Comparar o tempo de espera (minutos) em duas filas de banco. Os tempos são muito assimétricos, com alguns clientes esperando mais de 2 horas.

b) Medir a pressão arterial de 30 pacientes antes e depois de um remédio. As diferenças parecem normais.

c) Verificar se o peso médio de pacotes de café é 500 g, com uma amostra de 40 pacotes de distribuição aproximadamente normal.

d) Comparar a altura de jogadores de dois times. Os dois grupos parecem normais, mas o Levene deu p = 0,01.

e) Medir a relação entre a posição no ranking de um campeonato e o salário do jogador.

f) Mesma situação de (b), mas com só 8 pacientes e diferenças claramente não normais.

**Respostas:**

a) **Mann-Whitney.** São dois grupos independentes com dados muito assimétricos e outliers, e a média seria distorcida.

b) **t pareado** (`ttest_rel`). Os mesmos pacientes são medidos duas vezes e as diferenças são normais.

c) **t de uma amostra** (`ttest_1samp`, `popmean=500`). É uma média comparada a um valor fixo, com dados normais.

d) **t de Welch** (`equal_var=False`). Há normalidade, mas o Levene rejeitou variâncias iguais.

e) **Spearman.** Ranking é uma variável ordinal (posição) e a relação pode não ser linear.

f) **Wilcoxon** (`stats.wilcoxon`). É a versão não paramétrica do pareado, para amostra pequena sem normalidade.

## Simulado — análise completa

**Pergunta:** entre **apartamentos mobiliados** (`furniture == "furnished"`) e **não mobiliados** da **Zona Sul**, o aluguel é diferente?

Faça a análise completa, como numa prova:

1. Filtre os dados (só apartamentos da Zona Sul) e separe os dois grupos.
2. Faça um resumo (n, média, mediana, dp) e um gráfico comparando os grupos.
3. Escreva H₀ e H₁.
4. Cheque as premissas (normalidade com Q-Q plot e assimetria; variâncias com Levene).
5. Escolha e aplique o teste mais adequado. Se quiser, aplique o alternativo também para comparar.
6. Calcule um tamanho de efeito (diferença de medianas e/ou δ de Cliff).
7. Escreva a conclusão em 2–3 frases.

💡 Dica: com amostras grandes (milhares de imóveis), quase qualquer diferença dá p < 0,05. Por isso o **tamanho de efeito** é o que mostra se a diferença é relevante na prática.

In [ ]:
# 1. só apartamentos da Zona Sul, e os dois grupos
sul = imoveis.query("property_type == 'apartamento' and zona == 'Zona Sul'")
mob = sul.loc[sul["furniture"] == "furnished", "rent_amount"]
nao_mob = sul.loc[sul["furniture"] == "not furnished", "rent_amount"]
len(mob), len(nao_mob)

In [ ]:
# 2. resumo e gráfico
display(
    sul.groupby("furniture").rent_amount
    .agg(n="size", média="mean", mediana="median", dp="std")
    .round(0)
)

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.boxplot(data=sul, x="furniture", y="rent_amount", ax=ax)
ax.set(xlabel="", ylabel="Aluguel (R$)", xticks=[0, 1])
ax.set_xticklabels([t.get_text().replace("not furnished", "Não mobiliado").replace("furnished", "Mobiliado")
                    for t in ax.get_xticklabels()])
plt.tight_layout()
plt.show()

In [ ]:
# 3. H0: mobília não importa -> os aluguéis dos dois grupos têm a mesma distribuição
#    H1: um grupo tende a ter aluguéis diferentes do outro (bilateral)

In [ ]:
# 4. premissas
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, values, title in zip(axes, [mob, nao_mob], ["Mobiliado", "Não mobiliado"]):
    stats.probplot(values, dist="norm", plot=ax)
    ax.set(title=f"{title} · assimetria = {stats.skew(values):.2f}")
plt.tight_layout()
plt.show()

report("Levene", stats.levene(mob, nao_mob))

In [ ]:
# 5. aluguel é muito assimétrico -> Mann-Whitney é o principal; Welch só para comparar
mw = stats.mannwhitneyu(mob, nao_mob)
report("Mann-Whitney", mw, stat="U")
report("t de Welch", stats.ttest_ind(mob, nao_mob, equal_var=False), stat="t")

In [ ]:
# 6. tamanho de efeito
delta = 2 * mw.statistic / (len(mob) * len(nao_mob)) - 1
print(f"diferença de medianas = R$ {mob.median() - nao_mob.median():.0f}")
print(f"δ de Cliff = {delta:.2f}  ->  P(mobiliado mais caro) ≈ {(1 + delta) / 2:.2f}")

**Em palavras:** conclusão (2–3 frases).

O aluguel dos dois grupos é muito assimétrico (Q-Q plots curvados e assimetria alta) e as variâncias diferem (Levene p < 0,001). Por isso o **Mann-Whitney** é o teste principal. Ele rejeita H₀ (p < 0,001), assim como o Welch: na Zona Sul, apartamentos mobiliados tendem a ter aluguel maior, com mediana R$ 570 acima (R$ 3.690 contra R$ 3.120).

Porém o **δ de Cliff ≈ 0,11 é pequeno**: sorteando um mobiliado e um não mobiliado, o mobiliado é mais caro só em ~56% das vezes. O p-valor minúsculo vem principalmente do **tamanho da amostra** (~7.300 imóveis). A diferença é real, mas a mobília explica pouco do aluguel, e os dois grupos se sobrepõem muito.